## What is HuggingFace datasets Library?

A super-efficient, standardized data system for ML and NLP.

It handles loading, cleaning, streaming, caching, and column operations, fast.

It replaces messy manual code like:

downloading files, unzipping, parsing JSON/CSV/XML, cleaning rows, storing them, handling splits, batching, tokenizing efficiently

The library gives you:

🔹 A single object type: Dataset

🔹 A dictionary of splits: DatasetDict

🔹 Built-in tools for mapping, filtering, shuffling, batching

🔹 Automatic disk caching

🔹 Memory-efficient data formats (Apache Arrow)

🔹 Easy integration with PyTorch / TensorFlow / JAX

## The TWO main dataset objects: datasets and datasetdict

In [1]:
from datasets import load_dataset


ds = load_dataset("imdb")["train"]
print(ds)

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


This is like a Pandas DataFrame, but:

much faster

memory-efficient

optimized for ML workflows

immutable (you don’t modify in place; you create new datasets)

Each row is a Python dictionary:

In [5]:
print(ds[0]['text'])
print(ds[0]['label'])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

## DatasetDict → a dictionary of multiple datasets (splits)
test, train, unsupervised(unlabelled data) because labels are expensive

In [7]:
from datasets import load_dataset


ds = load_dataset("imdb")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [10]:
ds = load_dataset("imdb")['unsupervised']
print(ds[0])

{'text': 'This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything else, maybe except the movie "Leon" wonderfully played by Jean Reno and Natalie Portman. But... What can I say about this one? This is the best movie Anne Parillaud has ever played in (See please "Frankie Starlight", she\'s speaking English there) to see what I mean. The story of young punk girl Nikita, taken into the depraved world of the secret government forces has been exceptionally over used by Americans. Never mind the "Point of no return" and especially the "La femme Nikita" TV series. They cannot compare the original believe me! Trash these videos. Buy this one, do not rent it, BUY it. BTW beware of the subtitles of the LA company which "translate" the US release. What a disgrace! If you cant understand French, get a dubbed version. But you\'ll regret later :)', 'label': -1}


## Features: the schema

Features describe the data types of each column (is it a string? is it an integer)
FEATURES ARE IMPORTANT BECAUSE:

They determine how data is stored

They define how data is serialized (Arrow / Parquet)

They help model pipelines know what to expect

They guarantee consistency

| Feature                    | Meaning                                 |
| -------------------------- | --------------------------------------- |
| `Value("string")`          | plain text                              |
| `Value("int32")`           | integer                                 |
| `Value("float32")`         | float                                   |
| `ClassLabel`               | integer labels with names               |
| `Sequence(Value("int32"))` | list of ints                            |
| `Sequence` of `dicts`      | nested objects                          |
| `dict`                     | structured data (e.g., PubMed articles) |
| `Array2D`, `Array3D`       | images / embeddings                     |


In [11]:
ds.features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

## Key dataset operations

### 1. Indexing

In [16]:
ds[0] ## returns a dictionary for that row 
ds[0:5] ## returns a slice
print(len(ds[0:5]))

2


### 2. Filtering

But first small explanation of how lambda functionlaity works:

lambda row: row["label"] == 1

is the same thing as 

def lambda(row):

    return row["label"]==1
    

In [17]:
filtered = ds.filter(lambda row: row["label"] == 1) ## Efficient because:
                                                    # runs in parallel
                                                    # uses Apache Arrow
                                                    # auto-caches results

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

### 3. Mapping (the most important function)